# 03 — Exploratory Data Analysis of the 1-Minute SOLETE Data

## Purpose

The 60-minute and 5-minute notebooks established the basic structure of the
SOLETE data and highlighted several questions about measurement encoding and
daily-boundary behaviour. We now examine the 1-minute file to see whether those
patterns persist when less temporal aggregation is applied.

This notebook asks:

1. Is the 1-minute timeline complete and regularly sampled?
2. Are solar PV and wind power still plausible forecasting targets?
3. Do the pressure, humidity, and wind-direction issues seen at coarser
   resolutions remain visible?
4. Does the suspicious midnight behaviour persist at 1-minute resolution?
5. Are the main same-time target–driver relationships stable at this finer
   sampling rate?

No source value is modified in this notebook. The aim is to gather evidence
before choosing a modelling resolution or defining preprocessing rules.

## Data source

SOLETE contains meteorological measurements together with co-located wind and
solar photovoltaic generation from the SYSLAB facility in Denmark.

- Pombo, D. V., Gehrke, O., & Bindner, H. W. (2022), *SOLETE, a 15-month
  long holistic dataset including: Meteorology, co-located wind and solar PV
  power from Denmark with various resolutions*, Data in Brief, 42, 108046.
  DOI: `10.1016/j.dib.2022.108046`
- Dataset DOI: `10.11583/DTU.17040767`


In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

# Locate the project root whether the notebook is launched from the project
# directory or directly from notebooks/.
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_solete

DATASET_PATH = PROJECT_ROOT / "solete_dataset" / "SOLETE_Pombo_1min.h5"
if not DATASET_PATH.exists():
    raise FileNotFoundError(f"SOLETE file not found: {DATASET_PATH}")

df = load_solete(DATASET_PATH)


## Dataset and temporal structure

At 1-minute resolution the dataset is substantially larger, so the structural
checks are kept compact. Before any forecasting use, we verify chronology,
uniqueness, missing measurements, and the expected one-minute grid.


In [2]:
expected_index = pd.date_range(
    start=df.index.min(),
    end=df.index.max(),
    freq="1min",
)

time_differences = df.index.to_series().diff().dropna()

overview = pd.Series({
    "rows": len(df),
    "columns": df.shape[1],
    "start_timestamp": df.index.min(),
    "end_timestamp": df.index.max(),
    "timezone": str(df.index.tz),
    "missing_measurements": int(df.isna().sum().sum()),
    "duplicate_timestamps": int(df.index.duplicated().sum()),
    "chronological": df.index.is_monotonic_increasing,
    "expected_1min_rows": len(expected_index),
    "missing_1min_timestamps": len(expected_index.difference(df.index)),
    "non_1min_intervals": int(
        (time_differences != pd.Timedelta(minutes=1)).sum()
    ),
}, name="1-minute SOLETE")

display(overview.to_frame("value"))
display(df.head(3))


,value
rows,658081
columns,11
start_timestamp,2018-06-01 00:00:00+00:00
end_timestamp,2019-09-01 00:00:00+00:00
timezone,UTC
missing_measurements,0
duplicate_timestamps,0
chronological,True
expected_1min_rows,658081
missing_1min_timestamps,0


,TEMPERATURE[degC],HUMIDITY[%],WIND_SPEED[m1s],WIND_DIR[deg],GHI[kW1m2],POA Irr[kW1m2],P_Gaia[kW],P_Solar[kW],Pressure[mbar],Azimuth[deg],Elevation[deg]
Timestamp,,,,,,,,,,,
2018-06-01 00:00:00+00:00,14.600000,0.7,1.311864,108.135593,0.0,0.0,0.038733,0.0,1017.599976,0.0,0.0
2018-06-01 00:01:00+00:00,14.600000,0.7,1.548333,115.033333,0.0,0.0,0.000033,0.0,1017.599976,0.0,0.0
2018-06-01 00:02:00+00:00,14.596667,0.7,1.365000,104.683333,0.0,0.0,0.029767,0.0,1017.536658,0.0,0.0


## Candidate forecasting targets

As in the 5-minute data, the two renewable-generation variables are

- `P_Solar[kW]` — photovoltaic generation;
- `P_Gaia[kW]` — Gaia wind-turbine generation.

The finer data let us check whether both targets remain sufficiently populated
and whether short-lived extremes become more visible at 1-minute resolution.


In [3]:
target_columns = ["P_Solar[kW]", "P_Gaia[kW]"]

target_summary = pd.DataFrame({
    "mean_kW": df[target_columns].mean(),
    "median_kW": df[target_columns].median(),
    "max_kW": df[target_columns].max(),
    "zero_fraction": (df[target_columns] == 0).mean(),
    "nonzero_observations": (df[target_columns] > 0).sum(),
    "missing": df[target_columns].isna().sum(),
})

target_summary


,mean_kW,median_kW,max_kW,zero_fraction,nonzero_observations,missing
P_Solar[kW],0.988283,0.070645,8.404550,0.413998,385637,0
P_Gaia[kW],2.691753,0.852150,23.478949,0.343291,432168,0


## Measurement ranges and quality flags

We repeat only the quality checks that can influence later preprocessing.
These are diagnostic flags, not automatic deletion rules.

In particular, humidity is screened on its apparent fractional scale, wind
direction is checked as a circular variable, and pressure is checked for the
same implausibly high values seen at coarser resolutions.


In [4]:
descriptive = df.describe().T[
    ["count", "mean", "std", "min", "50%", "max"]
].rename(columns={"50%": "median"})

quality_flags = pd.Series({
    "humidity < 0 or > 1": int(
        ((df["HUMIDITY[%]"] < 0) | (df["HUMIDITY[%]"] > 1)).sum()
    ),
    "wind direction outside [0, 360)": int(
        ((df["WIND_DIR[deg]"] < 0) | (df["WIND_DIR[deg]"] >= 360)).sum()
    ),
    "negative GHI": int((df["GHI[kW1m2]"] < 0).sum()),
    "negative POA irradiance": int((df["POA Irr[kW1m2]"] < 0).sum()),
    "negative wind power": int((df["P_Gaia[kW]"] < 0).sum()),
    "negative solar power": int((df["P_Solar[kW]"] < 0).sum()),
    "pressure outside 800–1100 mbar": int(
        ((df["Pressure[mbar]"] < 800) | (df["Pressure[mbar]"] > 1100)).sum()
    ),
}, name="flagged observations")

display(descriptive)
display(quality_flags.to_frame())


,count,mean,std,min,median,max
TEMPERATURE[degC],658081.0,11.927159,7.213321,-4.980000,12.501667,44.600000
HUMIDITY[%],658081.0,0.768253,0.156192,0.000000,0.800000,2.000000
WIND_SPEED[m1s],658081.0,3.588490,2.144343,0.000000,3.263333,23.922034
WIND_DIR[deg],658081.0,208.943786,73.642975,0.000000,215.883333,680.000000
GHI[kW1m2],658081.0,0.140380,0.221207,0.000000,0.011017,1.291350
POA Irr[kW1m2],658081.0,0.144380,0.248869,0.000000,0.009000,1.187800
P_Gaia[kW],658081.0,2.691753,3.434584,0.000000,0.852150,23.478949
P_Solar[kW],658081.0,0.988283,1.688230,0.000000,0.070645,8.404550
Pressure[mbar],658081.0,1011.004293,27.981111,974.215009,1010.881684,2034.500000
Azimuth[deg],658081.0,-2.131793,57.476283,-153.220350,0.000000,146.237702


,flagged observations
humidity < 0 or > 1,164
"wind direction outside [0, 360)",103
negative GHI,0
negative POA irradiance,0
negative wind power,0
negative solar power,0
pressure outside 800–1100 mbar,456


## Daily-boundary behaviour

Notebook 02 showed that the abnormal pressure readings occur exactly at
00:00 UTC. If the same pattern exists in the 1-minute source, it cannot be
explained simply as an artifact created by 5-minute aggregation.

Gaia power is checked separately. The threshold of 12 kW is used only to
inspect the extreme tail; it is not a cleaning rule.


In [5]:
midnight = (df.index.hour == 0) & (df.index.minute == 0)
pressure_high = df["Pressure[mbar]"] > 1100
gaia_high = df["P_Gaia[kW]"] > 12

neighbor_pressure = (
    df["Pressure[mbar]"].shift(1) + df["Pressure[mbar]"].shift(-1)
) / 2

pressure_jump = (
    df.loc[midnight, "Pressure[mbar]"]
    - neighbor_pressure.loc[midnight]
).dropna()

boundary_summary = pd.Series({
    "pressure >1100 mbar": int(pressure_high.sum()),
    "pressure >1100 at midnight": int((pressure_high & midnight).sum()),
    "median midnight pressure jump (mbar)": pressure_jump.median(),
    "Gaia power >12 kW": int(gaia_high.sum()),
    "Gaia >12 kW at midnight": int((gaia_high & midnight).sum()),
    "maximum Gaia power at midnight (kW)": df.loc[midnight, "P_Gaia[kW]"].max(),
    "maximum Gaia power outside midnight (kW)": df.loc[~midnight, "P_Gaia[kW]"].max(),
}, name="daily-boundary diagnostics")

boundary_summary.to_frame("value")


,value
pressure >1100 mbar,456.000000
pressure >1100 at midnight,456.000000
median midnight pressure jump (mbar),1000.000000
Gaia power >12 kW,541.000000
Gaia >12 kW at midnight,34.000000
maximum Gaia power at midnight (kW),23.478949
maximum Gaia power outside midnight (kW),13.048600


## Target–driver relationships

The purpose of these correlations is to check whether the main physical
relationships seen at 5-minute resolution remain visible at 1-minute
resolution.

They are **same-timestamp descriptive relationships**, not forecasting
results. Future measured irradiance or weather values cannot later be treated
as known inputs unless they are genuinely available at forecast issue time.


In [6]:
target_driver_relationships = pd.Series({
    "PV vs POA irradiance": df["P_Solar[kW]"].corr(df["POA Irr[kW1m2]"]),
    "PV vs GHI": df["P_Solar[kW]"].corr(df["GHI[kW1m2]"]),
    "PV vs source elevation": df["P_Solar[kW]"].corr(df["Elevation[deg]"]),
    "PV vs humidity": df["P_Solar[kW]"].corr(df["HUMIDITY[%]"]),
    "Wind power vs wind speed": df["P_Gaia[kW]"].corr(df["WIND_SPEED[m1s]"]),
}, name="Pearson correlation")

source_elevation_summary = pd.Series({
    "minimum elevation (deg)": df["Elevation[deg]"].min(),
    "maximum elevation (deg)": df["Elevation[deg]"].max(),
    "zero-elevation observations": int((df["Elevation[deg]"] == 0).sum()),
    "nonzero PV while elevation == 0": int(
        ((df["P_Solar[kW]"] > 0) & (df["Elevation[deg]"] == 0)).sum()
    ),
}, name="source elevation")

display(target_driver_relationships.to_frame())
display(source_elevation_summary.to_frame("value"))


,Pearson correlation
PV vs POA irradiance,0.995232
PV vs GHI,0.913345
PV vs source elevation,0.743143
PV vs humidity,-0.599584
Wind power vs wind speed,0.816795


,value
minimum elevation (deg),0.000000
maximum elevation (deg),58.265999
zero-elevation observations,307051.000000
nonzero PV while elevation == 0,52449.000000


## Findings

The 1-minute SOLETE file contains **658,081 observations** across the same 11
variables. It covers the same study period as the coarser files, contains no
missing measurements or duplicate timestamps, is already chronological, and
follows a complete one-minute grid.

### Renewable-generation targets

Both `P_Solar[kW]` and `P_Gaia[kW]` remain available as candidate forecasting
targets. The finer sampling preserves considerably more short-term variation
than the hourly representation and provides a useful view of brief wind-power
extremes.

### Data-quality observations

Several patterns seen at 5-minute resolution persist:

- **103 wind-direction observations** exceed the conventional 360-degree
  boundary, reinforcing the need for circular interpretation;
- **456 pressure observations** exceed 1100 mbar;
- all 456 high-pressure observations occur exactly at **00:00 UTC**;
- humidity again behaves mainly like a fractional series but reaches values
  above 1.

Because the pressure anomaly exists directly in the 1-minute file, it is not
simply created by 5-minute aggregation. Its highly regular timing supports
further investigation as a source-data boundary artifact.

The Gaia tail requires more care. There are **541 observations above 12 kW**,
but only **34** occur exactly at midnight. The maximum outside midnight is
about **13.05 kW**, whereas the midnight maximum reaches about **23.48 kW**.
Therefore, a simple rule such as “all values above 12 kW are invalid” would
not be justified by this EDA.

### Target–driver relationships

The main same-time relationships remain close to those seen at 5-minute
resolution:

- PV vs POA irradiance: approximately **0.995**;
- PV vs GHI: approximately **0.913**;
- PV vs source elevation: approximately **0.743**;
- PV vs humidity: approximately **−0.600**;
- wind power vs wind speed: approximately **0.817**.

This stability suggests that the major physical relationships are preserved
when moving from 5-minute to 1-minute sampling, while the finer data retain
additional short-term variability.

These correlations remain descriptive only; they do not determine which
future measurements can legally be used in a forecasting model.

No source observation is modified in this notebook.

## Next step

The 1-second SOLETE file will be examined next using a deliberately lightweight
and memory-conscious analysis. Once all available resolutions have been
understood, they can be compared directly before selecting the modelling
resolution and defining preprocessing rules.
